# Run Debug API

Use this notebook to turn Langfuse traces plus TestPrompt oracles into per-run debugging data. The API persists `run.debug`, then exposes rows and summaries that are easy to load into pandas or hand to another agent.

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "packages" / "research").exists() and (candidate / "packages" / "pipeline").exists():
        ROOT = candidate
        break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from datetime import datetime, timezone
from typing import Any
from dotenv import load_dotenv

from packages.research.experiment import Experiment, ExperimentRun, ReproEnvelope, RunDebugAPI, TaskRegistry
from packages.research.experiment.run_debug import LangfuseTraceReader
from packages.research.storage.store import SQLiteStore
from packages.research.strategy.protocol import StrategyConfig

load_dotenv(ROOT / ".env", override=False)
print("Langfuse env loaded:", bool(os.getenv("LANGFUSE_PUBLIC_KEY")), bool(os.getenv("LANGFUSE_SECRET_KEY")))
print("Langfuse host:", os.getenv("LANGFUSE_HOST") or os.getenv("LANGFUSE_BASE_URL"))


Langfuse env loaded: True True
Langfuse host: https://us.cloud.langfuse.com


## 1. Inspect The TestPrompt Oracles

`TaskRegistry` now keeps one expected RoboGrammar structure per prompt. These are the answer keys used to count TP, FP, FN, and TN-like negatives when you provide a node universe.

In [2]:
store = SQLiteStore()
registry = TaskRegistry(store=store)

for item in registry.list()[:5]:
    print(item["label"], "|", item["prompt"])
    print("  expected:", item["expected_robo_grammar"])


morphlogy-provided | quadruped robot that can climb stairs
  expected: {'S': ['BODY', 'CLIMBING_LEG_PAIR', 'CLIMBING_LEG_PAIR', 'TAIL']}
infer-morphology-single | robot that should choose one best body plan for narrow warehouse aisles
  expected: {'S': ['BODY']}
infer-morphology-family | robot family for mixed terrain search and rescue
  expected: {'S': ['BODY', 'LEG_PAIR', 'WHEEL_PAIR', 'BODY_CHAIN', 'STABILIZER']}
open-ended | invent a robot for moving fragile objects across clutter
  expected: {'S': ['BODY_CHAIN', 'BODY_SEG', 'BODY_SEG', 'BODY_SEG', 'MANIPULATION_MODULE']}
morphlogy-provided | quadruped robot with four articulated legs for climbing outdoor stairs while carrying a small sensor pack
  expected: {'S': ['BODY', 'CLIMBING_LEG_PAIR', 'CLIMBING_LEG_PAIR', 'TAIL', 'MANIPULATION_MODULE', 'PAYLOAD_BAY', 'SENSOR_MAST']}


## 2. Debug A Langfuse Cloud Trace Directly

The repo `.env` is loaded above. `LangfuseTraceReader` reads `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, and either `LANGFUSE_HOST` or `LANGFUSE_BASE_URL`. Use `last_run_id()`, `last_k_run_ids(k)`, or `all_run_ids()` to pull Langfuse trace/run IDs newest-first. `debug_trace` infers the prompt from the graph/root observation input, so the trace ID is enough for normal RoboGrammar traces. Use `use_codex=True` for agent-written trace analysis; the API resolves `CODEX_CLI`, the shell `PATH`, Homebrew, and common NVM install paths so notebooks do not depend on a login-shell `PATH`.

In [4]:
cloud_api = RunDebugAPI(store=store, trace_reader=LangfuseTraceReader(env_path=ROOT / ".env"))

# Pull Langfuse trace/run IDs directly from Cloud, newest first.
latest_trace_id = cloud_api.last_run_id()
recent_trace_ids = cloud_api.last_k_run_ids(5)
print("latest trace:", latest_trace_id)
recent_trace_ids
# all_trace_ids = cloud_api.all_run_ids()  # paginates every matching observation row
# recent_refs = cloud_api.langfuse_run_refs(limit=5)  # includes timestamps and observation names

# Verify debug_trace across every discovered Langfuse run without Codex analysis.
# all_trace_ids = cloud_api.all_run_ids()
# cloud_reports = [cloud_api.debug_trace(trace_id) for trace_id in all_trace_ids]
# print("debugged", len(cloud_reports), "runs")
# cloud_api.summarize(cloud_reports)

# Debug the latest run, or paste a Langfuse trace ID from the UI.
# This does not require a local ExperimentRun row.

# cloud_report = cloud_api.debug_trace(latest_trace_id, use_codex=True)
# cloud_report

batch_trace_ids = cloud_api.last_k_run_ids(3)
batch_report = cloud_api.batch_debug_trace(batch_trace_ids)
batch_report

latest trace: 2a1d820dc4b24603f6ef2cfd2082101a


KeyboardInterrupt: 

## 3. Batch Debug Langfuse Traces

`batch_debug_trace(trace_ids)` fetches each trace, builds deterministic per-run reports, then asks Codex to analyze the whole batch in one pass. Use this when you want cross-run failure patterns instead of one report per trace.

In [ ]:
# Keep batches small at first because Codex receives all selected trace context.
batch_trace_ids = cloud_api.last_k_run_ids(3)
# batch_report = cloud_api.batch_debug_trace(batch_trace_ids)
# batch_report


## 4. Debug A Persisted Run

New grammar runs store `langfuse_trace_id` when Langfuse tracing is enabled. Use this path after `ExperimentRunner.run(...)` or `run_suite(...)` has persisted a run.

In [ ]:
# debugged_run = cloud_api.debug_run("<local-run-id>", use_codex=True)
# debugged_run.debug


## 5. Local Example Without Langfuse Credentials

This reproduces the failure mode where the rule builder creates four climbing leg pairs for a quadruped task and the evaluator incorrectly passes it.

In [ ]:
class InMemoryTraceReader:
    def fetch_trace(self, trace_id: str) -> dict[str, Any]:
        return {
            "trace_id": trace_id,
            "observations": [
                {
                    "name": "build_structural_rules",
                    "type": "AGENT",
                    "output": {
                        "structural_rules": {
                            "S": [
                                "BODY",
                                "CLIMBING_LEG_PAIR",
                                "CLIMBING_LEG_PAIR",
                                "CLIMBING_LEG_PAIR",
                                "CLIMBING_LEG_PAIR",
                                "TAIL",
                            ]
                        }
                    },
                },
                {
                    "name": "evaluate_rules",
                    "type": "EVALUATOR",
                    "output": {
                        "criteria": [
                            {
                                "description": "Rules include a quadruped climbing proxy.",
                                "isSuccessful": True,
                                "critiques": [],
                            }
                        ]
                    },
                },
            ],
        }

demo_store = SQLiteStore(db_path=ROOT / "packages" / "research" / ".runs" / "debug_demo.db")
demo_store.save_experiment(Experiment(experiment_id="debug-demo", name="debug-demo"))
demo_prompt = registry.register("quadruped robot that can climb stairs", "morphlogy-provided")
demo_store.save_test_prompt(demo_prompt)

run = ExperimentRun(
    run_id="demo-run",
    experiment_id="debug-demo",
    strategy_name="grammar",
    prompt=demo_prompt["prompt"],
    config=StrategyConfig(seed=1, model_id="test", max_candidates=1),
    repro=ReproEnvelope(seed=1, model_id="test", prompt_hash="demo", strategy_name="grammar", strategy_version="1"),
    started_at=datetime.now(timezone.utc),
    finished_at=datetime.now(timezone.utc),
    prompt_label=demo_prompt["label"],
    langfuse_trace_id="demo-trace",
)
demo_store.save_run(run)

api = RunDebugAPI(store=demo_store, trace_reader=InMemoryTraceReader())
debugged = api.debug_run("demo-run")
debugged.debug


## 6. Turn Reports Into Analysis Tables

These are the dataframe-friendly surfaces for questions like: which pipeline nodes fail most, how often does the evaluator pass bad traces, and whether evaluator proxy criteria mention task-relevant structure.

In [ ]:
report = debugged.debug

print(api.summarize([report]))
print("\nNode failures:")
for row in api.node_failure_rows([report]):
    print(row)

print("\nProxy checks:")
for row in report.proxy_checks:
    print(row)


## 7. Experiment-Level Debugging

Run this after an experiment has completed and Langfuse trace IDs are present. `only_failed=True` keeps the output focused on runs with oracle mismatches or node-level failures.

In [ ]:
# exp = store.find_experiment_by_name("notebook-demo")
# api = RunDebugAPI(store=store, trace_reader=LangfuseTraceReader(env_path=ROOT / ".env"))
# reports = api.debug_experiment(exp.experiment_id, only_failed=False, use_codex=False)
# api.summarize(reports)
# api.node_failure_rows(reports)[:10]
